In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import fdrcorrection
from collections import Counter

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils import resample

# ==========================================
# 1. DATA IMPORT & PREPARATION
# ==========================================
raw_df = pd.read_csv("golub.csv", index_col=0).copy()
label_map = {"allB": 0, "allT": 0, "aml": 1}
raw_df["target"] = raw_df["cancer"].map(label_map)

metadata_cols = ["Samples", "BM.PB", "Gender", "Source", "tissue.mf", "cancer", "target"]
feature_cols = [column for column in raw_df.columns if column not in metadata_cols]
X = raw_df[feature_cols]
y = raw_df["target"]

# Convert to NumPy arrays for faster computation
X_array = X.values
y_array = y.values

# ==========================================
# 2. SETUP NESTED CROSS-VALIDATION
# ==========================================
# Outer Loop: 5 Folds to test generalization
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Inner Loop: 3 Folds for Hyperparameter Tuning (GridSearch) inside the training set
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
param_grid = {'C': np.linspace(0.1, 0.8, 20)} # Slightly reduced grid for speed

# Metrics tracking
fold_metrics = []
all_selected_biomarkers = []

print("Starting Nested Cross-Validation Pipeline...\n")

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_array, y_array)):
    print(f"--- Processing Outer Fold {fold + 1}/5 ---")
    
    # 2a. Split Data
    X_train, X_test = X_array[train_idx], X_array[test_idx]
    y_train, y_test = y_array[train_idx], y_array[test_idx]
    
    # 2b. Scale Data (Fit on Train, Transform on Test to prevent data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ==========================================
    # 3. PHASE 2: STATISTICAL BASELINE (On Training Data Only)
    # ==========================================
    # Isolate ALL and AML patients in the training set
    X_train_ALL = X_train_scaled[y_train == 0]
    X_train_AML = X_train_scaled[y_train == 1]
    
    # Fast vectorized T-test
    t_stat, p_vals_t = ttest_ind(X_train_ALL, X_train_AML, equal_var=False, axis=0)
    _, p_adj_t = fdrcorrection(np.nan_to_num(p_vals_t, nan=1.0), alpha=0.01)
    sig_t = set(np.array(feature_cols)[p_adj_t < 0.01])
    
    # Mann-Whitney U test (Iterative but fast enough for 7k features)
    p_vals_mw = [mannwhitneyu(X_train_ALL[:, i], X_train_AML[:, i]).pvalue for i in range(X_train_scaled.shape[1])]
    _, p_adj_mw = fdrcorrection(p_vals_mw, alpha=0.01)
    sig_mw = set(np.array(feature_cols)[p_adj_mw < 0.01])
    
    # Robust Baseline Intersection
    robust_baseline_genes = sig_t.intersection(sig_mw)

    # ==========================================
    # 4. PHASE 3: GRID SEARCH & FEATURE SELECTION
    # ==========================================
    # Optimize Lasso C parameter using the Inner CV
    lasso_model = LogisticRegression(solver='saga', l1_ratio=1.0, max_iter=10000, random_state=42)
    grid_search = GridSearchCV(lasso_model, param_grid, cv=inner_cv, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train_scaled, y_train)
    optimal_C = grid_search.best_params_['C']

    # ==========================================
    # 5. PHASE 4: STABILITY ANALYSIS (BOOTSTRAPPING)
    # ==========================================
    B = 100
    boot_lasso = LogisticRegression(solver='saga', l1_ratio=1.0, C=optimal_C, max_iter=10000, random_state=42)
    fold_selected_genes = []
    
    for i in range(B):
        # Resample training data
        X_boot, y_boot = resample(X_train_scaled, y_train, random_state=i)
        boot_lasso.fit(X_boot, y_boot)
        
        # Extract non-zero genes
        nonzero_idx = np.where(boot_lasso.coef_[0] != 0)[0]
        fold_selected_genes.extend([feature_cols[idx] for idx in nonzero_idx])
        
    # Calculate Inclusion Probabilities and filter >= 80%
    gene_counts = Counter(fold_selected_genes)
    highly_stable_genes = [gene for gene, count in gene_counts.items() if (count / B) >= 0.80]
    all_selected_biomarkers.extend(highly_stable_genes)

    # ==========================================
    # 6. PHASE 5: FINAL EVALUATION ON HOLD-OUT TEST SET
    # ==========================================
    if len(highly_stable_genes) == 0:
        print("Warning: No stable genes found in this fold. Skipping evaluation.")
        continue
        
    # Map stable gene names back to numerical indices
    final_indices = [feature_cols.index(g) for g in highly_stable_genes]
    
    # Slice the scaled train/test data to only include the stable genes
    X_train_final = X_train_scaled[:, final_indices]
    X_test_final = X_test_scaled[:, final_indices]
    
    # Train the final unpenalized (L2) model
    final_model = LogisticRegression(C=1.0, solver='lbfgs', random_state=42)
    final_model.fit(X_train_final, y_train)
    y_pred = final_model.predict(X_test_final)
    
    # Log Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    concordance = len(set(highly_stable_genes).intersection(robust_baseline_genes)) / len(highly_stable_genes)
    
    fold_metrics.append({
        'Fold': fold + 1,
        'Optimal_C': optimal_C,
        'Stable_Genes': len(highly_stable_genes),
        'Concordance': concordance,
        'Accuracy': acc,
        'F1_Score': f1
    })
    
    print(f"  -> Stable Genes: {len(highly_stable_genes)}")
    print(f"  -> Concordance: {concordance*100:.1f}% | Accuracy: {acc*100:.1f}%\n")

# ==========================================
# 7. FINAL SYNTHESIS & REPORTING
# ==========================================
results_df = pd.DataFrame(fold_metrics)

print("==========================================")
print(" FINAL NESTED CROSS-VALIDATION RESULTS")
print("==========================================")
print(results_df.to_string(index=False))

print("\n--- Averages Across All 5 Folds ---")
print(f"Average Stable Genes per Fold: {results_df['Stable_Genes'].mean():.1f}")
print(f"Average Concordance Rate:      {results_df['Concordance'].mean()*100:.1f}%")
print(f"Average Test Accuracy:         {results_df['Accuracy'].mean()*100:.1f}%")
print(f"Average Test F1-Score:         {results_df['F1_Score'].mean():.4f}")

# Identify the "Super-Biomarkers" (Genes selected in multiple folds)
global_gene_counts = Counter(all_selected_biomarkers)
print("\n--- Global 'Super-Biomarkers' (Selected in >= 3 Folds) ---")
for gene, count in global_gene_counts.most_common():
    if count >= 3:
        print(f"{gene}: Selected in {count}/5 Folds")

Starting Nested Cross-Validation Pipeline...

--- Processing Outer Fold 1/5 ---
  -> Stable Genes: 9
  -> Concordance: 100.0% | Accuracy: 100.0%

--- Processing Outer Fold 2/5 ---
  -> Stable Genes: 6
  -> Concordance: 100.0% | Accuracy: 100.0%

--- Processing Outer Fold 3/5 ---
  -> Stable Genes: 14
  -> Concordance: 100.0% | Accuracy: 100.0%

--- Processing Outer Fold 4/5 ---
  -> Stable Genes: 10
  -> Concordance: 100.0% | Accuracy: 85.7%

--- Processing Outer Fold 5/5 ---
  -> Stable Genes: 1
  -> Concordance: 100.0% | Accuracy: 100.0%

 FINAL NESTED CROSS-VALIDATION RESULTS
 Fold  Optimal_C  Stable_Genes  Concordance  Accuracy  F1_Score
    1   0.763158             9          1.0  1.000000       1.0
    2   0.542105             6          1.0  1.000000       1.0
    3   0.800000            14          1.0  1.000000       1.0
    4   0.689474            10          1.0  0.857143       0.8
    5   0.247368             1          1.0  1.000000       1.0

--- Averages Across All 5 Fol